In [17]:
import pandas as pd

df=pd.read_csv("heart_disease.csv")

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [18]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  target    303 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 33.3 KB
None


In [19]:
print(df.describe())

              age         sex          cp  ...          ca        thal      target
count  303.000000  303.000000  303.000000  ...  299.000000  301.000000  303.000000
mean    54.438944    0.679868    3.158416  ...    0.672241    4.734219    0.458746
std      9.038662    0.467299    0.960126  ...    0.937438    1.939706    0.499120
min     29.000000    0.000000    1.000000  ...    0.000000    3.000000    0.000000
25%     48.000000    0.000000    3.000000  ...    0.000000    3.000000    0.000000
50%     56.000000    1.000000    3.000000  ...    0.000000    3.000000    0.000000
75%     61.000000    1.000000    4.000000  ...    1.000000    7.000000    1.000000
max     77.000000    1.000000    4.000000  ...    3.000000    7.000000    1.000000

[8 rows x 14 columns]


In [20]:
print(df.isnull().sum())

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64


In [21]:
print(df["target"].value_counts())

target
0    164
1    139
Name: count, dtype: int64


In [22]:
from sklearn.model_selection import train_test_split
x=df.drop("target",axis=1)
y=df["target"]

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [23]:
#handling missing values
ca_mode=x_train["ca"].mode()[0]
thal_mode=x_train["thal"].mode()[0]

x_train["ca"]=x_train["ca"].fillna(ca_mode)
x_test["ca"]=x_test["ca"].fillna(ca_mode)
x_train["thal"]=x_train["thal"].fillna(thal_mode)
x_test["thal"]=x_test["thal"].fillna(thal_mode)

In [24]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [25]:
from sklearn.linear_model import LogisticRegression

lr=LogisticRegression()
lr.fit(x_train,y_train)
y_pred_lr=lr.predict(x_test)
lr_probs = lr.predict_proba(x_test)[:, 1]

In [26]:
from sklearn.tree import DecisionTreeClassifier

dt=DecisionTreeClassifier(max_depth=5,random_state=42)
dt.fit(x_train,y_train)
y_pred_dt=dt.predict(x_test)
dt_probs=dt.predict_proba(x_test)[:,1]

In [27]:
from sklearn.ensemble import RandomForestClassifier

rf=RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(x_train,y_train)
y_pred_rf=rf.predict(x_test)
rf_probs=rf.predict_proba(x_test)[:,1]

In [33]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score

results = pd.DataFrame(
    {
        "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
        "Accuracy": [
            accuracy_score(y_test, y_pred_lr),
            accuracy_score(y_test, y_pred_dt),
            accuracy_score(y_test, y_pred_rf),
        ],
        "ROC-AUC": [
            roc_auc_score(y_test, lr_probs),
            roc_auc_score(y_test, dt_probs),
            roc_auc_score(y_test, rf_probs),
        ],
        "Precision": [
            precision_score(y_test, y_pred_lr),
            precision_score(y_test, y_pred_dt),
            precision_score(y_test, y_pred_rf),
        ],
        "Recall":[
            recall_score(y_test,y_pred_lr),
            recall_score(y_test,y_pred_dt),
            recall_score(y_test,y_pred_rf),
        ],
        "F1 score":[
          f1_score(y_test,y_pred_lr),
          f1_score(y_test,y_pred_dt),
          f1_score(y_test,y_pred_rf),  
        ]
    }
)

print(results)

                 Model  Accuracy   ROC-AUC  Precision   Recall  F1 score
0  Logistic Regression  0.885246  0.921336   0.878788  0.90625  0.892308
1        Decision Tree  0.721311  0.716056   0.758621  0.68750  0.721311
2        Random Forest  0.868852  0.931573   0.900000  0.84375  0.870968
